# PortPy `models_debug.ipynb` -- Full Package Function Reference

A point-by-point exercise of every public, callable function and class in `portpy.models`
(`estimators`, `optimization`, `construction`, `management`, plus `ModelResult` and the
`Portfolio.models` auto-fill mechanics), run against real market data via `yfinance`,
mirroring `metrics_debug.ipynb`'s structure and coverage-audit mechanism.

Every section is numbered `N.M` and maps to one function (or one closely related group of
calls on the same function). Section 6 cross-checks call coverage (every function actually
invoked) and explain coverage (every function's `Explanation` card actually rendered)
against `portpy.models`' own `__all__` lists, then brute-force-renders every registered
`"model"`/relevant `"function"` card to confirm none of them raise.

In [1]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
import yfinance as yf

import portpy
from portpy import Portfolio
from portpy import explain as pexplain
from portpy.explain import available, get
from portpy.models.base import GroupCap, GrossExposure, ModelResult, NetExposure, TurnoverCap, WeightBounds

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

print("portpy version:", portpy.__version__)

# Tracks every function/class actually called below (section 6.1) and every one whose
# Explanation card was actually rendered (section 6.2) - the same mechanism
# metrics_debug.ipynb uses, adapted for portpy.models' name collisions across submodules
# (both optimization and construction export a distinct function named "efficient_frontier").
COVERED: set[str] = set()
EXPLAINED: set[str] = set()


def note(*names: str) -> None:
    COVERED.update(names)


def note_explained(*names: str) -> None:
    EXPLAINED.update(names)

portpy version: 0.2.0


## 1. Setup and Data

In [2]:
# Five assets spanning equity/bond/commodity, plus a market benchmark - enough variety
# for covariance shrinkage, factor regressions, and constraints to be meaningful without
# needing dozens of tickers. Calendar-alignment/mixed-timeframe handling (crypto, different
# currencies) is metrics_debug.ipynb's job, not this one's - every asset here trades Mon-Fri.
TICKERS = ["AAPL", "MSFT", "JPM", "TLT", "GLD"]
BENCHMARK = "SPY"

raw = yf.download(TICKERS + [BENCHMARK], period="4y", auto_adjust=True, progress=False)["Close"]
raw = raw.dropna(how="any")
prices = raw[TICKERS]
benchmark_returns = raw[BENCHMARK].pct_change().dropna()

portfolio = Portfolio(prices, name="Debug Portfolio", risk_free_rate=0.02)
print(portfolio)
print("weights:", portfolio.weights.round(3).to_dict())

Portfolio(name='Debug Portfolio', assets=5, n_obs=1002, frequency=252)
weights: {'AAPL': 0.2, 'MSFT': 0.2, 'JPM': 0.2, 'TLT': 0.2, 'GLD': 0.2}


## 2. `portpy.models.estimators`

### 2.1 `expected_returns` -- all four methods

In [3]:
from portpy.models.estimators.expected_returns import expected_returns

y = portfolio.asset_returns()

mu_hist = expected_returns(y, method="mean_historical")
print("mean_historical (arithmetic):\n", mu_hist.round(4))

mu_hist_geo = expected_returns(y, method="mean_historical", geometric=True)
print("\nmean_historical (geometric):\n", mu_hist_geo.round(4))

mean_historical (arithmetic):
 Ticker
AAPL    0.2386
MSFT    0.2274
JPM     0.3262
TLT    -0.0216
GLD     0.2541
Name: expected_return, dtype: float64

mean_historical (geometric):
 AAPL    0.2235
MSFT    0.2094
JPM     0.3478
TLT    -0.0323
GLD     0.2638
Name: expected_return, dtype: float64


In [4]:
mu_ewma = expected_returns(y, method="ewma", span=90)
print("ewma (span=90):\n", mu_ewma.round(4))

mu_capm = expected_returns(y, method="capm_implied", benchmark=benchmark_returns, rf=portfolio.risk_free_rate)
print("\ncapm_implied:\n", mu_capm.round(4))

ewma (span=90):
 Ticker
AAPL    0.5240
MSFT    0.7894
JPM     0.2144
TLT    -0.1474
GLD    -0.0633
Name: expected_return, dtype: float64

capm_implied:
 AAPL   0.2177
MSFT   0.2080
JPM    0.1734
TLT    0.0449
GLD    0.0614
Name: expected_return, dtype: float64


In [5]:
# james_stein's shrinkage intensity phi=(N+2)/((N+2)+T*mahalanobis) is computed at the
# data's own periodic scale (T = number of daily observations, mahalanobis on periodic
# mu/cov) - independent of periods_per_year, so it shrinks meaningfully whenever the
# assets' historical means are close together relative to their covariance, without
# needing a synthetic example or a short lookback window to make the effect visible.
mu_js = expected_returns(y, method="james_stein")
implied_phi = 1.0 - float((mu_js - mu_hist.mean()).abs().sum() / (mu_hist - mu_hist.mean()).abs().sum())
print(f"james_stein (implied phi~{implied_phi:.4f}):")
print(pd.DataFrame({"naive": mu_hist, "james_stein": mu_js}).round(4))
print("\ngrand mean (james_stein's default shrinkage target):", round(float(mu_hist.mean()), 4))
assert (mu_js - mu_hist.mean()).abs().sum() < (mu_hist - mu_hist.mean()).abs().sum()

pexplain("expected_returns", value=mu_hist)
note("expected_returns")
note_explained("expected_returns")

james_stein (implied phi~0.3877):
         naive  james_stein
Ticker                     
AAPL    0.2386       0.2255
MSFT    0.2274       0.2187
JPM     0.3262       0.2792
TLT    -0.0216       0.0662
GLD     0.2541       0.2350

grand mean (james_stein's default shrinkage target): 0.2049
expected_returns (function)

What it is:
  Estimates the expected annual return of each asset, the mu vector that every optimizer in .models.optimization treats as its return forecast.

Formula:
  mean_historical: mean(y)*periods_per_year | ewma: ewm(span).mean()*periods_per_year | capm_implied: rf + beta*(E[market]-rf) | james_stein: shrink mean(y) toward the grand mean by phi=(N+2)/((N+2)+T*mahalanobis)

How to read it:
  One annualized return number per asset, in the same units as any other annual return figure (0.08 = 8%/yr). Feed it straight into .models.optimization.* as `expected_returns=`.

Good vs. bad:
  There's no 'good value' here - the question is whether the *method* fits your use case.

### 2.2 `covariance` -- all five methods, checked for positive semi-definiteness

In [6]:
from portpy.models.estimators.covariance import covariance

for method in ("sample", "ewma", "shrinkage", "ledoit_wolf", "robust"):
    cov_m = covariance(y, method=method)
    eigvals = np.linalg.eigvalsh(cov_m.to_numpy())
    print(f"{method:12s} shape={cov_m.shape}  min_eigval={eigvals.min():.6f}  (PSD: {eigvals.min() > -1e-8})")

cov = covariance(y, method="ledoit_wolf")
print("\nledoit_wolf covariance (annualized):\n", cov.round(4))

sample       shape=(5, 5)  min_eigval=0.020023  (PSD: True)
ewma         shape=(5, 5)  min_eigval=0.007894  (PSD: True)
shrinkage    shape=(5, 5)  min_eigval=0.020927  (PSD: True)


ledoit_wolf  shape=(5, 5)  min_eigval=0.021862  (PSD: True)


robust       shape=(5, 5)  min_eigval=0.016340  (PSD: True)

ledoit_wolf covariance (annualized):
 Ticker   AAPL   MSFT     JPM     TLT    GLD
Ticker                                     
AAPL   0.0727 0.0311  0.0186  0.0042 0.0034
MSFT   0.0311 0.0738  0.0151  0.0013 0.0046
JPM    0.0186 0.0151  0.0550 -0.0013 0.0032
TLT    0.0042 0.0013 -0.0013  0.0241 0.0054
GLD    0.0034 0.0046  0.0032  0.0054 0.0402


In [7]:
raw_sample = covariance(y, method="sample", annualize=False)
annualized_sample = covariance(y, method="sample", annualize=True, periods_per_year=252)
print("annualize=True is exactly the periodic matrix * 252:")
print((annualized_sample / raw_sample).round(2))  # every entry should read 252.0

# "shrinkage" targets a diagonal-of-sample matrix (each asset keeps its own variance) -
# NOT a scaled-identity (equal-variance) target the way some other libraries' "shrinkage"
# does; see the covariance() docstring and the cross-validation notebook for the ~1% gap
# this produces against PyPortfolioOpt's shrunk_covariance() at the same nominal intensity.
shrunk = covariance(y, method="shrinkage", shrinkage_intensity=1.0)
sample = covariance(y, method="sample")
print("\nshrinkage_intensity=1.0 keeps each asset's own variance (diagonal matches sample exactly):")
print(pd.DataFrame({"sample_diag": np.diag(sample), "shrunk_diag (intensity=1.0)": np.diag(shrunk)}, index=cov.columns).round(4))
assert np.allclose(np.diag(shrunk.to_numpy()), np.diag(sample.to_numpy()))
assert np.allclose(shrunk.to_numpy() - np.diag(np.diag(shrunk.to_numpy())), 0.0)  # off-diagonal is exactly zero

pexplain("covariance", value=cov)
note("covariance")
note_explained("covariance")

annualize=True is exactly the periodic matrix * 252:
Ticker     AAPL     MSFT      JPM      TLT      GLD
Ticker                                             
AAPL   252.0000 252.0000 252.0000 252.0000 252.0000
MSFT   252.0000 252.0000 252.0000 252.0000 252.0000
JPM    252.0000 252.0000 252.0000 252.0000 252.0000
TLT    252.0000 252.0000 252.0000 252.0000 252.0000
GLD    252.0000 252.0000 252.0000 252.0000 252.0000

shrinkage_intensity=1.0 keeps each asset's own variance (diagonal matches sample exactly):
        sample_diag  shrunk_diag (intensity=1.0)
Ticker                                          
AAPL         0.0740                       0.0740
MSFT         0.0751                       0.0751
JPM          0.0552                       0.0552
TLT          0.0224                       0.0224
GLD          0.0395                       0.0395
covariance (function)

What it is:
  Estimates the asset return covariance matrix (Sigma) that every optimizer in .models.optimization uses to measu

### 2.3 `factor_models` -- linear_regression, rolling_regression, capm, fama_french, factor_attribution

In [8]:
from portpy.models.estimators.factor_models import (
    capm,
    factor_attribution,
    fama_french,
    linear_regression,
    rolling_regression,
)

capm_model = capm(portfolio.returns(), benchmark_returns, rf=portfolio.risk_free_rate)
print(f"capm: alpha(const)={capm_model.params['const']:.6f}  beta(market)={capm_model.params['market']:.4f}  R2={capm_model.rsquared:.4f}")
pexplain("capm", value=capm_model)
note("capm")
note_explained("capm")

capm: alpha(const)=0.000253  beta(market)=0.6950  R2=0.6807
capm (model)

What it is:
  The Capital Asset Pricing Model: explains an asset's excess return purely through its sensitivity (beta) to the market's excess return.

Formula:
  (y - rf) = alpha + beta*(benchmark - rf) + e

How to read it:
  alpha ("const") is the average return unexplained by market exposure, per period - a positive, statistically significant alpha is the classic (if often overstated/data-mined) signature of genuine skill. beta is the market sensitivity, matching metrics.risk.beta's Cov/Var formula when rf=0 - see the cross-validation tests for the exact relationship at rf != 0.

Good vs. bad:
  A high R-squared means the single-factor market model explains most of the return variation, so alpha/beta are estimated with more confidence; a low R-squared means most of the return is idiosyncratic and the CAPM lens isn't telling you much.

Caveats:
  CAPM is a one-factor simplification - real returns are also driven

In [9]:
from portpy.metrics.risk import beta as metrics_beta

# capm's beta (from an OLS regression) and metrics.risk.beta (Cov/Var closed form) are two
# independent formulas for the same quantity - they should match closely at rf=0.
capm_at_zero_rf = capm(portfolio.returns(), benchmark_returns, rf=0.0)
print("capm beta:        ", round(float(capm_at_zero_rf.params["market"]), 6))
print("metrics.risk.beta:", round(float(metrics_beta(portfolio.returns(), benchmark_returns)), 6))
assert np.isclose(capm_at_zero_rf.params["market"], metrics_beta(portfolio.returns(), benchmark_returns), atol=1e-8)

capm beta:         0.695049
metrics.risk.beta: 0.695049


In [10]:
roll = rolling_regression(portfolio.returns(), benchmark_returns.rename("market"), window=126)
print("rolling_regression shape:", roll.shape, "columns:", list(roll.columns))
print(roll.dropna().tail())
pexplain("rolling_regression", value=roll)
note("rolling_regression")
note_explained("rolling_regression")

rolling_regression shape: (1001, 2) columns: ['alpha', 'market']
             alpha  market
Date                      
2026-09-08 -0.0000  0.6952
2026-09-09  0.0000  0.6932
2026-09-10  0.0001  0.6886
2026-09-11  0.0001  0.6856
2026-09-14  0.0002  0.6798
rolling_regression (function)

What it is:
  Runs linear_regression repeatedly over a sliding window, showing how alpha and factor loadings evolve through time instead of assuming they're constant.

Formula:
  Same as linear_regression, refit independently on each window of `window` consecutive observations.

How to read it:
  Each row is a snapshot regression using only the trailing `window` periods - a rolling beta drifting from 0.8 to 1.3 means the asset's market sensitivity genuinely changed, not that the single-window estimate was wrong.

Good vs. bad:
  Stable rolling coefficients support treating the single full-sample regression as reliable; large swings mean the 'true' beta/alpha is time-varying and a single static regression i

In [11]:
# Illustrative style factors (not real Ken French data): a size-like factor from
# AAPL-vs-JPM and a rates-like factor from TLT's own return, just to exercise the
# multi-factor machinery end to end.
factors = pd.DataFrame({
    "Mkt-RF": benchmark_returns - portfolio.risk_free_rate / 252,
    "SMB": (portfolio.asset_returns()["AAPL"] - portfolio.asset_returns()["JPM"]).reindex(benchmark_returns.index),
    "HML": portfolio.asset_returns()["TLT"].reindex(benchmark_returns.index),
}).dropna()

ff_model = fama_french(portfolio.returns(), factors, version=3)
print(f"fama_french: R2={ff_model.rsquared:.4f}")
print(ff_model.params.round(6))
pexplain("fama_french", value=ff_model)
note("fama_french")
note_explained("fama_french")

try:
    fama_french(portfolio.returns(), factors.rename(columns={"SMB": "not_smb"}), version=3)
except ValueError as exc:
    print(f"\nExpected ValueError for a missing canonical column: {exc}")

fama_french: R2=0.7272
const    0.0004
Mkt-RF   0.6613
SMB      0.0346
HML      0.1746
dtype: float64
fama_french (model)

What it is:
  Extends CAPM with additional priced factors (size, value, and optionally profitability/investment) to explain more of an asset's return than the market factor alone.

Formula:
  y[- rf if an RF column is supplied] = alpha + beta_Mkt*Mkt-RF + beta_SMB*SMB + beta_HML*HML [+ beta_RMW*RMW + beta_CMA*CMA] + e

How to read it:
  Each beta is the asset's loading on that factor - e.g. a positive SMB beta means the asset behaves like a small-cap stock, a positive HML beta means it behaves like a value stock. alpha is what's left unexplained by all factors together, usually smaller (in magnitude) than a single-factor CAPM alpha on the same asset.

Good vs. bad:
  A smaller, less significant alpha than the CAPM alpha on the same asset means the extra factors are absorbing return that looked like 'skill' under CAPM but is really a known style exposure. Large, sig

In [12]:
attribution = factor_attribution(portfolio.returns(), factors)
print(attribution.round(6))
print("\ncontributions + alpha sum to the mean return:", attribution["contribution"].sum().round(8), "vs", portfolio.returns().mean().round(8))
pexplain("factor_attribution", value=attribution)
note("factor_attribution")
note_explained("factor_attribution")

         beta  contribution  pct_of_total
Mkt-RF 0.6613        0.0005        0.5623
SMB    0.0346       -0.0000       -0.0148
HML    0.1746       -0.0000       -0.0184
alpha     NaN        0.0004        0.4709

contributions + alpha sum to the mean return: 0.00081317 vs 0.00081317
factor_attribution (function)

What it is:
  Splits an asset's average historical return into how much came from each factor's average level versus how much is unexplained (alpha).

Formula:
  contribution_i = beta_i * mean(factor_i); alpha = const; total = sum(contribution_i) + alpha

How to read it:
  pct_of_total shows each factor's share of the asset's average return - a factor can have a large beta but a small contribution if that factor's own average return was near zero over the sample.

Good vs. bad:
  A return mostly attributed to well-known factors (high combined pct_of_total on Mkt/size/value/etc.) means the strategy is largely a repackaged factor bet; a return mostly attributed to 'alpha' means it

In [13]:
lr = linear_regression(portfolio.returns(), factors)
print(f"linear_regression: R2={lr.rsquared:.4f}, params={dict(lr.params.round(4))}")
pexplain("linear_regression", value=lr)
note("linear_regression")
note_explained("linear_regression")

linear_regression: R2=0.7272, params={'const': np.float64(0.0004), 'Mkt-RF': np.float64(0.6613), 'SMB': np.float64(0.0346), 'HML': np.float64(0.1746)}
linear_regression (function)

What it is:
  General-purpose OLS regression with an intercept - the shared engine behind capm, fama_french, and factor_attribution.

Formula:
  y = const + beta_1*x_1 + ... + beta_k*x_k + e, fit by ordinary least squares

How to read it:
  `.params` holds the intercept ("const") and each regressor's coefficient; `.rsquared` is the fraction of y's variance explained; `.pvalues` flags which coefficients are statistically distinguishable from zero.

Good vs. bad:
  Higher R-squared means the regressors explain more of y's variation - not necessarily a 'better' model for decision-making, since a high R-squared factor exposure you don't want is still a risk, not a feature.

Caveats:
  Assumes linearity, i.i.d. Normal-ish errors, and no perfect multicollinearity among regressors (statsmodels will warn or fail on 

## 3. `portpy.models.optimization` -- pure solver functions

Every function takes `(expected_returns, cov_matrix, ...)` (or just `cov_matrix` for the
return-agnostic ones) and returns a plain `pd.Series` of weights - no `Portfolio`
dependency at all.

In [14]:
from portpy.models import optimization as opt

mu = mu_hist

w_mv = opt.mean_variance(mu, cov, risk_aversion=2.0)
print("mean_variance:\n", w_mv.round(3))
assert abs(w_mv.sum() - 1.0) < 1e-6
pexplain("mean_variance", value=ModelResult(name="mean_variance", weights=w_mv))
note("mean_variance")
note_explained("mean_variance")

mean_variance:
 AAPL   0.0000
MSFT   0.0000
JPM    0.8230
TLT    0.0000
GLD    0.1770
Name: weight, dtype: float64
mean_variance (model)

What it is:
  Markowitz's original portfolio problem: pick weights that maximize expected return net of a risk penalty on variance.

Formula:
  maximize w'mu - 0.5*risk_aversion*w'Sigma*w, s.t. sum(w)=1 (+ any extra constraints)

How to read it:
  Higher risk_aversion pulls the solution toward the global minimum-variance portfolio; lower risk_aversion pulls it toward concentrating in the highest-mu assets.

Good vs. bad:
  A well-diversified solution (moderate HHI, several active positions) suggests mu/Sigma disagree enough to justify holding multiple assets; a solution collapsed into 1-2 assets often means mu is dominating an under-diversified Sigma estimate (Michaud's 'error maximization').

Caveats:
  Extremely sensitive to the expected_returns input - see expected_returns' own caveats. risk_aversion has no universal 'correct' value; it must be ca

In [15]:
from portpy.metrics.covariance import portfolio_volatility

w_minvar = opt.min_variance(cov)
print("min_variance:\n", w_minvar.round(3))
print("\nmin_variance volatility:", round(portfolio_volatility(w_minvar, cov), 4))
print("mean_variance volatility:", round(portfolio_volatility(w_mv, cov), 4), "(should be >= min_variance's)")
assert portfolio_volatility(w_mv, cov) >= portfolio_volatility(w_minvar, cov) - 1e-9
pexplain("min_variance", value=ModelResult(name="min_variance", weights=w_minvar))
note("min_variance")
note_explained("min_variance")

min_variance:
 AAPL   0.0490
MSFT   0.0880
JPM    0.1830
TLT    0.4620
GLD    0.2180
Name: weight, dtype: float64

min_variance volatility: 0.1114
mean_variance volatility: 0.1985 (should be >= min_variance's)
min_variance (model)

What it is:
  The single portfolio on the efficient frontier with the lowest possible variance, ignoring expected returns entirely.

Formula:
  minimize w'Sigma*w, s.t. sum(w)=1 (+ any extra constraints)

How to read it:
  This is the leftmost point of the efficient frontier - every other frontier portfolio has equal or higher risk.

Good vs. bad:
  Appropriate when you have little confidence in your return forecasts (a common, defensible choice, since Sigma is estimated far more reliably than mu) or explicitly want the lowest-risk fully-invested portfolio.

Caveats:
  Says nothing about return - the min-variance portfolio can have a very low or even negative expected return. Still inherits covariance estimation-risk caveats from whatever cov_matrix estimato

In [16]:
w_sharpe = opt.max_sharpe(mu, cov, rf=portfolio.risk_free_rate)
print("max_sharpe:\n", w_sharpe.round(3))
realized_sharpe = (w_sharpe.to_numpy() @ mu.to_numpy() - portfolio.risk_free_rate) / portfolio_volatility(w_sharpe, cov)
print("\nrealized Sharpe at these weights:", round(realized_sharpe, 3))
pexplain("max_sharpe", value=ModelResult(name="max_sharpe", weights=w_sharpe))
note("max_sharpe")
note_explained("max_sharpe")

max_sharpe:
 AAPL   0.0940
MSFT   0.0890
JPM    0.3820
TLT    0.0000
GLD    0.4360
Name: weight, dtype: float64

realized Sharpe at these weights: 1.76
max_sharpe (model)

What it is:
  The portfolio on the efficient frontier with the highest ratio of expected excess return to volatility - the tangency portfolio.

Formula:
  maximize (w'mu - rf) / sqrt(w'Sigma*w), s.t. sum(w)=1 (+ any extra constraints)

How to read it:
  Geometrically, this is where a line from the risk-free rate is tangent to the efficient frontier - combining it with cash/leverage traces out the entire capital allocation line.

Good vs. bad:
  A high in-sample Sharpe here is close to guaranteed by construction (it's literally what's being maximized) - it says little about out-of-sample performance. Judge the *inputs* (mu/Sigma quality), not the achieved objective value.

Caveats:
  The Sharpe-ratio objective is non-convex in general, unlike mean_variance/min_variance - the multi-start solve in portpy.models.base exi

In [17]:
# Target chosen above the global min-variance return - see target_return's own docstring
# for why a target below it still pins you there exactly (an equality, not ">= target").
min_var_return = float(w_minvar.to_numpy() @ mu.to_numpy())
target = min_var_return + 0.03
w_tr = opt.target_return(mu, cov, target=target)
print(f"target_return(target={target:.4f}):\n", w_tr.round(3))
print("realized return:", round(float(w_tr.to_numpy() @ mu.to_numpy()), 4))
print("diagnostics:", w_tr.attrs["solve_diagnostics"])
assert w_tr.attrs["solve_diagnostics"]["success"]
pexplain("target_return", value=ModelResult(name="target_return", weights=w_tr, diagnostics=w_tr.attrs["solve_diagnostics"]))
note("target_return")
note_explained("target_return")

target_return(target=0.1668):
 AAPL   0.0610
MSFT   0.0910
JPM    0.2200
TLT    0.3620
GLD    0.2660
Name: weight, dtype: float64
realized return: 0.1668
diagnostics: {'success': True, 'objective_value': 0.01280794145783172, 'iterations': 11, 'message': 'Optimization terminated successfully', 'n_restarts': 4}
target_return (model)

What it is:
  The lowest-variance portfolio that achieves an exact target expected return - one specific point on the efficient frontier.

Formula:
  minimize w'Sigma*w, s.t. w'mu = target, sum(w)=1 (+ any extra constraints)

How to read it:
  Use this when you have a required return (e.g. a liability or spending target) rather than a risk-aversion preference.

Good vs. bad:
  diagnostics['success']=False after solving means `target` was infeasible given the other constraints (e.g. above the return of the single best asset, or below what's reachable with a WeightBounds floor) - treat the returned weights as unreliable in that case, not as a valid frontier po

In [18]:
min_vol = portfolio_volatility(w_minvar, cov)
target_vol = min_vol * 1.3
w_tv = opt.target_volatility(mu, cov, target=target_vol)
print(f"target_volatility(target={target_vol:.4f}):\n", w_tv.round(3))
print("realized volatility:", round(portfolio_volatility(w_tv, cov), 4))

# Deliberately infeasible target (below the achievable minimum volatility) - should NOT
# raise, but should flag non-convergence in diagnostics rather than silently lying.
w_infeasible = opt.target_volatility(mu, cov, target=min_vol * 0.1)
print("\ninfeasible target (below min_vol) diagnostics:", w_infeasible.attrs["solve_diagnostics"])
assert not w_infeasible.attrs["solve_diagnostics"]["success"]

pexplain("target_volatility", value=ModelResult(name="target_volatility", weights=w_tv))
note("target_volatility")
note_explained("target_volatility")

target_volatility(target=0.1448):
 AAPL   0.1030
MSFT   0.1010
JPM    0.3510
TLT    0.0030
GLD    0.4410
Name: weight, dtype: float64
realized volatility: 0.1448



infeasible target (below min_vol) diagnostics: {'success': False, 'objective_value': -0.1368277264361955, 'iterations': 37, 'message': 'Positive directional derivative for linesearch', 'n_restarts': 4}
target_volatility (model)

What it is:
  The highest-expected-return portfolio subject to a volatility cap - the frontier point at a chosen risk level.

Formula:
  maximize w'mu, s.t. w'Sigma*w <= target^2, sum(w)=1 (+ any extra constraints)

How to read it:
  Use this when you have a risk budget (e.g. 'no more than 12% annualized volatility') rather than a risk-aversion preference.

Good vs. bad:
  The cap almost always binds exactly (realized volatility == target) whenever target is above the global minimum-variance portfolio's volatility; if target is below it, the problem is infeasible and diagnostics['success'] will read False.

Caveats:
  Same mu-sensitivity caveat as mean_variance; the cap constraint is convex (a quadratic inequality) but the overall multi-start SLSQP path is sti

In [19]:
frontier = opt.efficient_frontier(mu, cov, n_points=20)
print(frontier[["target_return", "return", "volatility", "sharpe"]].round(4))
assert frontier["volatility"].diff().dropna().ge(-1e-6).all(), "volatility should be non-decreasing along the frontier"
pexplain("efficient_frontier", value=ModelResult(name="efficient_frontier", weights=frontier.iloc[-1][TICKERS], diagnostics={"frontier": frontier}))
note("optimization.efficient_frontier")
note_explained("efficient_frontier")

    target_return  return  volatility  sharpe
0          0.1368  0.1368      0.1114  1.2287
1          0.1468  0.1468      0.1116  1.3158
2          0.1568  0.1568      0.1122  1.3976
3          0.1667  0.1667      0.1132  1.4733
4          0.1767  0.1767      0.1145  1.5425
5          0.1866  0.1866      0.1163  1.6050
6          0.1966  0.1966      0.1184  1.6606
7          0.2066  0.2066      0.1208  1.7095
8          0.2165  0.2165      0.1236  1.7520
9          0.2265  0.2265      0.1267  1.7884
10         0.2365  0.2365      0.1300  1.8193
11         0.2464  0.2464      0.1336  1.8451
12         0.2564  0.2564      0.1374  1.8664
13         0.2664  0.2664      0.1414  1.8837
14         0.2763  0.2763      0.1457  1.8970
15         0.2863  0.2863      0.1531  1.8701
16         0.2963  0.2963      0.1650  1.7953
17         0.3062  0.3062      0.1820  1.6823
18         0.3162  0.3162      0.2058  1.5363
19         0.3262  0.3262      0.2345  1.3909
efficient_frontier (model)

What i

In [20]:
w_hrp = opt.hierarchical_risk_parity(cov)
print("hierarchical_risk_parity:\n", w_hrp.round(3))
assert abs(w_hrp.sum() - 1.0) < 1e-6
assert (w_hrp >= -1e-9).all()
pexplain("hierarchical_risk_parity", value=ModelResult(name="hierarchical_risk_parity", weights=w_hrp))
note("hierarchical_risk_parity")
note_explained("hierarchical_risk_parity")

hierarchical_risk_parity:
 AAPL   0.1160
MSFT   0.1410
JPM    0.1540
TLT    0.3680
GLD    0.2210
Name: weight, dtype: float64
hierarchical_risk_parity (model)

What it is:
  Lopez de Prado's (2016) HRP: clusters assets by correlation structure, then allocates risk recursively down the resulting tree instead of inverting the full covariance matrix.

Formula:
  quasi-diagonalize Sigma via hierarchical clustering on correlation distance sqrt(0.5*(1-corr)); recursively split inverse-variance weight between the two halves of each cluster

How to read it:
  No risk_aversion or expected_returns to tune - the only real lever is linkage_method, which changes how assets get clustered.

Good vs. bad:
  Tends to spread weight more evenly across correlated groups than mean-variance-based methods, and is far less sensitive to small changes in the covariance matrix (no matrix inversion) - the tradeoff is that it optimizes nothing explicitly, so it can be dominated in-sample by a well-specified mean-v

In [21]:
from portpy.metrics.covariance import component_contribution_to_risk

w_rp = opt.risk_parity(cov)
risk_contrib = component_contribution_to_risk(w_rp.reindex(cov.columns), cov)
print("risk_parity weights:\n", w_rp.round(3))
print("\nper-asset risk contribution (should be roughly equal):", np.round(risk_contrib, 5))
assert np.ptp(risk_contrib) < 1e-3
pexplain("risk_parity", value=ModelResult(name="risk_parity", weights=w_rp))
note("risk_parity")
note_explained("risk_parity")

risk_parity weights:
 AAPL   0.1390
MSFT   0.1450
JPM    0.1840
TLT    0.3060
GLD    0.2250
Name: weight, dtype: float64

per-asset risk contribution (should be roughly equal): [0.02366 0.02366 0.02366 0.02366 0.02366]
risk_parity (model)

What it is:
  Equal Risk Contribution: allocates so that every asset contributes the same share of total portfolio volatility, rather than the same dollar weight.

Formula:
  solve for w such that w_i * (Sigma@w)_i is equal across all i, s.t. sum(w)=1, w>=0

How to read it:
  A low-volatility asset ends up with a *larger* weight than a high-volatility one, specifically so their risk contributions match - don't compare the weights to a cap-weighted or equal-weight benchmark and expect them to look similar.

Good vs. bad:
  Nearly-equal component_contribution_to_risk values (see metrics.covariance) across assets confirms the solve converged correctly; large residual dispersion after solving means diagnostics['success'] is probably False.

Caveats:
  Lo

In [22]:
budget = pd.Series({"AAPL": 2.0, "MSFT": 1.0, "JPM": 1.0, "TLT": 1.0, "GLD": 1.0})
w_budget = opt.risk_budgeting(cov, budget)
budget_risk_contrib = component_contribution_to_risk(w_budget.reindex(cov.columns), cov)
print("risk_budgeting (AAPL wants 2x the risk share):\n", w_budget.round(3))
print("\nrisk contributions:", np.round(budget_risk_contrib, 5), "(AAPL's should be ~2x each of the other four's)")
pexplain("risk_budgeting", value=ModelResult(name="risk_budgeting", weights=w_budget))
note("risk_budgeting")
note_explained("risk_budgeting")

risk_budgeting (AAPL wants 2x the risk share):
 AAPL   0.2090
MSFT   0.1280
JPM    0.1660
TLT    0.2850
GLD    0.2120
Name: weight, dtype: float64

risk contributions: [0.04103 0.02052 0.02052 0.02052 0.02052] (AAPL's should be ~2x each of the other four's)
risk_budgeting (model)

What it is:
  Generalizes risk_parity to an explicit, non-equal target split of total portfolio risk across assets.

Formula:
  solve for w such that w_i * (Sigma@w)_i / portfolio_variance is proportional to budget_i, s.t. sum(w)=1, w>=0

How to read it:
  budget is renormalized to sum to 1 internally, so only relative sizes matter - budget=[2,1,1] means the first asset should carry twice the risk share of each of the other two.

Good vs. bad:
  Realized risk contributions (metrics.covariance.component_contribution_to_risk) close to the requested budget shares confirm convergence; large deviations mean check diagnostics['success'].

Caveats:
  Same long-only-by-construction and expected-returns-agnostic cavea

In [23]:
from portpy.metrics.covariance import diversification_ratio

w_md = opt.maximum_diversification(cov)
print("maximum_diversification:\n", w_md.round(3))
print("\nachieved diversification_ratio:", round(diversification_ratio(w_md, cov), 3))
print("equal-weight diversification_ratio:  ", round(diversification_ratio(pd.Series(0.2, index=cov.columns), cov), 3))
assert diversification_ratio(w_md, cov) >= diversification_ratio(pd.Series(0.2, index=cov.columns), cov) - 1e-6
pexplain("maximum_diversification", value=ModelResult(name="maximum_diversification", weights=w_md))
note("maximum_diversification")
note_explained("maximum_diversification")

maximum_diversification:
 AAPL   0.0980
MSFT   0.1320
JPM    0.1960
TLT    0.3430
GLD    0.2310
Name: weight, dtype: float64

achieved diversification_ratio: 1.811
equal-weight diversification_ratio:   1.712
maximum_diversification (model)

What it is:
  Choueifaty & Coignard's (2008) Most Diversified Portfolio: maximizes metrics.covariance.diversification_ratio directly.

Formula:
  maximize (w'sigma) / sqrt(w'Sigma*w), s.t. sum(w)=1 (+ any extra constraints)

How to read it:
  The result is, by construction, the fully-invested portfolio with the highest possible diversification_ratio given Sigma - call metrics.covariance.diversification_ratio on the resulting weights to see the achieved value.

Good vs. bad:
  A diversification_ratio well above 1 (see that metric's own card) confirms the optimizer found real diversification benefit in the correlation structure; a value near 1 means the assets are too correlated for this method to add much over equal-risk alternatives.

Caveats:
  Ign

In [24]:
market_weights = pd.Series(1 / len(TICKERS), index=TICKERS)

# No views -> should reproduce market_weights almost exactly at the same risk_aversion.
w_bl_no_views = opt.black_litterman(cov, market_weights, views={}, risk_aversion=2.5)
print("black_litterman, no views (should match market_weights):\n", w_bl_no_views.round(4))
assert np.allclose(w_bl_no_views.reindex(TICKERS).to_numpy(), market_weights.to_numpy(), atol=1e-3)

# A bullish view on GLD should pull its weight up relative to the no-views baseline.
w_bl_bullish = opt.black_litterman(cov, market_weights, views={"GLD": 0.15}, risk_aversion=2.5)
print("\nblack_litterman, bullish GLD view:\n", w_bl_bullish.round(4))
print("GLD weight: no-views =", round(w_bl_no_views["GLD"], 4), " bullish =", round(w_bl_bullish["GLD"], 4))
assert w_bl_bullish["GLD"] > w_bl_no_views["GLD"]

pexplain("black_litterman", value=ModelResult(name="black_litterman", weights=w_bl_bullish))
note("black_litterman")
note_explained("black_litterman")

black_litterman, no views (should match market_weights):
 AAPL   0.2000
MSFT   0.2000
JPM    0.2000
TLT    0.2000
GLD    0.2000
Name: weight, dtype: float64

black_litterman, bullish GLD view:
 AAPL   0.1558
MSFT   0.1378
JPM    0.0756
TLT    0.0000
GLD    0.6309
Name: weight, dtype: float64
GLD weight: no-views = 0.2  bullish = 0.6309
black_litterman (model)

What it is:
  Blends market-implied equilibrium returns with your own explicit views (with confidence levels), producing a posterior return estimate that's typically far more stable than plugging historical means straight into mean-variance.

Formula:
  pi = risk_aversion*Sigma@w_mkt; posterior_mu = [(tau*Sigma)^-1 + P'Omega^-1P]^-1 [(tau*Sigma)^-1 pi + P'Omega^-1 Q]; then mean_variance(posterior_mu, Sigma, risk_aversion)

How to read it:
  Without any views (an empty views dict), posterior_mu collapses back to pi exactly, which reproduces market_weights exactly at the same risk_aversion used to build pi - views only pull the res

### 3.1 Constraints applied directly to a solver function

In [25]:
w_bounded = opt.mean_variance(mu, cov, constraints=[WeightBounds(low=0.0, high=0.35)])
print("mean_variance with a 35% cap per asset:\n", w_bounded.round(3))
assert (w_bounded <= 0.35 + 1e-6).all()

mean_variance with a 35% cap per asset:
 AAPL   0.2750
MSFT   0.0250
JPM    0.3500
TLT    0.0000
GLD    0.3500
Name: weight, dtype: float64


In [26]:
w_grouped = opt.mean_variance(
    mu, cov, constraints=[GroupCap(groups={"equity": ["AAPL", "MSFT", "JPM"]}, max_weight=0.5)]
)
print("mean_variance with a 50% equity-group cap:\n", w_grouped.round(3))
print("equity exposure:", round(w_grouped[["AAPL", "MSFT", "JPM"]].sum(), 4))
assert w_grouped[["AAPL", "MSFT", "JPM"]].sum() <= 0.5 + 1e-6

mean_variance with a 50% equity-group cap:
 AAPL   0.0000
MSFT   0.0000
JPM    0.5000
TLT    0.0000
GLD    0.5000
Name: weight, dtype: float64
equity exposure: 0.5


In [27]:
# TurnoverCap.max_turnover caps the RAW sum(|w - current|) - NOT divided by 2, unlike
# metrics.costs.turnover_from_weights's "one-way turnover" convention (which halves the
# same sum). A cap of 0.10 here therefore allows up to 5% one-way turnover, not 10%.
current = pd.Series(0.2, index=TICKERS)
w_turnover = opt.mean_variance(mu, cov, constraints=[TurnoverCap(max_turnover=0.10, current_weights=current)])
raw_sum = float((w_turnover - current).abs().sum())
one_way = raw_sum / 2.0
print("mean_variance with a TurnoverCap(max_turnover=0.10):\n", w_turnover.round(3))
print(f"\nrealized sum(|delta w|) = {raw_sum:.4f}  (<= the 0.10 cap)")
print(f"equivalent one-way turnover = {one_way:.4f}  (half of the above - see TurnoverCap's own docstring)")
assert raw_sum <= 0.10 + 1e-6

mean_variance with a TurnoverCap(max_turnover=0.10):
 AAPL   0.2000
MSFT   0.2000
JPM    0.2500
TLT    0.1500
GLD    0.2000
Name: weight, dtype: float64

realized sum(|delta w|) = 0.1000  (<= the 0.10 cap)
equivalent one-way turnover = 0.0500  (half of the above - see TurnoverCap's own docstring)


In [28]:
# Negative weights (short-selling): WeightBounds(low<0) plus GrossExposure allow a
# long/short book instead of the default long-only, fully-invested one.
w_long_short = opt.mean_variance(
    mu, cov, constraints=[WeightBounds(low=-0.3, high=0.6), GrossExposure(1.3)], risk_aversion=0.5
)
print("mean_variance, long/short book (130% gross):\n", w_long_short.round(3))
print("net exposure:", round(w_long_short.sum(), 4), " gross exposure:", round(w_long_short.abs().sum(), 4))
assert np.isclose(w_long_short.abs().sum(), 1.3, atol=1e-6)
assert (w_long_short >= -0.3 - 1e-6).all() and (w_long_short <= 0.6 + 1e-6).all()

mean_variance, long/short book (130% gross):
 AAPL    0.1000
MSFT    0.0000
JPM     0.6000
TLT    -0.0000
GLD     0.6000
Name: weight, dtype: float64
net exposure: 1.3  gross exposure: 1.3


## 4. `portpy.models.construction`

### 4.1 Constraint objects (re-exported from `portpy.models.base`)

In [29]:
from portpy.models.construction import constraints as construction_constraints

print("construction.constraints re-exports:", construction_constraints.__all__)

instances = {
    "WeightBounds": construction_constraints.WeightBounds(low=0.0, high=0.4),
    "GroupCap": construction_constraints.GroupCap(groups={"equity": ["AAPL", "MSFT"]}, max_weight=0.5),
    "TurnoverCap": construction_constraints.TurnoverCap(max_turnover=0.1, current_weights=portfolio.weights),
    "NetExposure": construction_constraints.NetExposure(target=1.0),   # applied automatically by every call above unless overridden
    "GrossExposure": construction_constraints.GrossExposure(target=1.3),
}
for cls_name, instance in instances.items():
    print(f"{cls_name:14s}: {instance}")
    note(cls_name)

construction.constraints re-exports: ['WeightBounds', 'GroupCap', 'TurnoverCap', 'NetExposure', 'GrossExposure']
WeightBounds  : WeightBounds(low=0.0, high=0.4, per_asset=None)
GroupCap      : GroupCap(groups={'equity': ['AAPL', 'MSFT']}, max_weight=0.5, min_weight=None)
TurnoverCap   : TurnoverCap(max_turnover=0.1, current_weights=AAPL   0.2000
MSFT   0.2000
JPM    0.2000
TLT    0.2000
GLD    0.2000
Name: weight, dtype: float64)
NetExposure   : NetExposure(target=1.0)
GrossExposure : GrossExposure(target=1.3)


### 4.2 `optimize` -- the discoverable dispatcher

In [30]:
for method in (
    "mean_variance", "max_sharpe", "min_variance", "target_return", "target_volatility",
    "black_litterman", "hierarchical_risk_parity", "risk_parity", "risk_budgeting", "maximum_diversification",
):
    kwargs = {}
    if method == "target_return":
        kwargs["target"] = target
    elif method == "target_volatility":
        kwargs["target"] = min_vol * 1.2
    elif method == "black_litterman":
        kwargs["market_weights"] = market_weights
        kwargs["views"] = {"AAPL": 0.15}
    elif method == "risk_budgeting":
        kwargs["budget"] = pd.Series(1.0, index=TICKERS)
    result = portfolio.models.optimize(method=method, **kwargs)
    print(f"{method:26s} n_active={int((result.weights.abs() > 0.01).sum())}  converged={result.diagnostics.get('success', 'n/a')}")

result = portfolio.models.optimize(method="max_sharpe")
print("\nrepr:", repr(result))
print("\nsummary():\n", result.summary())
note("optimize")

mean_variance              n_active=1  converged=True
max_sharpe                 n_active=4  converged=True
min_variance               n_active=5  converged=True
target_return              n_active=5  converged=True
target_volatility          n_active=5  converged=True
black_litterman            n_active=5  converged=True
hierarchical_risk_parity   n_active=5  converged=n/a
risk_parity                n_active=5  converged=True
risk_budgeting             n_active=5  converged=True


maximum_diversification    n_active=5  converged=True



repr: ModelResult(name='max_sharpe', n_assets=5, n_active=4, converged=True)

summary():
       weight
GLD   0.4488
JPM   0.3745
AAPL  0.0900
MSFT  0.0867
TLT   0.0000


### 4.3 `build` -- estimate, then optimize, in one call

In [31]:
built = portfolio.models.build(method="mean_variance", covariance_kwargs={"method": "ledoit_wolf"})
print(repr(built))
print("built_from:", built.meta["built_from"])

# hierarchical_risk_parity needs no expected_returns - build() should skip that estimation step.
built_hrp = portfolio.models.build(method="hierarchical_risk_parity")
print("\n", repr(built_hrp))
print("expected_returns_kwargs recorded (should be empty - HRP is return-agnostic):", built_hrp.meta["built_from"]["expected_returns_kwargs"])
assert built_hrp.meta["built_from"]["expected_returns_kwargs"] == {}
note("build")

ModelResult(name='mean_variance', n_assets=5, n_active=1, converged=True)
built_from: {'expected_returns_kwargs': {}, 'covariance_kwargs': {'method': 'ledoit_wolf'}, 'n_assets': 5, 'n_observations': 1001}

 ModelResult(name='hierarchical_risk_parity', n_assets=5, n_active=5)
expected_returns_kwargs recorded (should be empty - HRP is return-agnostic): {}


### 4.4 `efficient_frontier` (construction layer) -- wrapped into a `ModelResult`

In [32]:
ef_result = portfolio.models.efficient_frontier(n_points=15)
print(repr(ef_result))
print("\n.weights is the max-Sharpe point on the curve:\n", ef_result.weights.round(3))
print("\n.summary() returns the full curve:\n", ef_result.summary().head())
note("construction.efficient_frontier")

ModelResult(name='efficient_frontier', n_assets=5, n_active=5)

.weights is the max-Sharpe point on the curve:
 AAPL   0.0950
MSFT   0.0950
JPM    0.3470
TLT    0.0180
GLD    0.4450
Name: weight, dtype: float64

.summary() returns the full curve:
    target_return  return  volatility  sharpe   AAPL   MSFT    JPM    TLT    GLD
0         0.1300  0.1300      0.1103  1.1785 0.0392 0.0838 0.1816 0.4874 0.2080
1         0.1440  0.1440      0.1107  1.3010 0.0448 0.0850 0.1981 0.4404 0.2317
2         0.1580  0.1580      0.1118  1.4131 0.0504 0.0861 0.2146 0.3935 0.2554
3         0.1720  0.1720      0.1137  1.5130 0.0560 0.0873 0.2311 0.3466 0.2790
4         0.1860  0.1860      0.1163  1.6001 0.0616 0.0884 0.2476 0.2996 0.3027


## 5. `portpy.models.management`

### 5.1 `rebalance`

In [33]:
current_weights = pd.Series(0.2, index=TICKERS)
target_result = portfolio.models.optimize(method="max_sharpe")

results_by_method = {}
for method in ("threshold", "full", "calendar"):
    kwargs = {"threshold": 0.05} if method == "threshold" else {}
    rebal = portfolio.models.management.rebalance(target_weights=target_result, method=method, cost_bps=10, **kwargs)
    results_by_method[method] = rebal
    print(f"{method:10s} n_traded={rebal.diagnostics['n_traded']}  turnover={rebal.diagnostics['turnover']:.2%}  est_cost={rebal.diagnostics['estimated_cost']:.4%}")

# "full" and "calendar" are documented as identical (rebalance() is a stateless,
# point-in-time trade-list generator, not a scheduler) - confirm that's actually true.
pd.testing.assert_series_equal(results_by_method["full"].weights, results_by_method["calendar"].weights)
print("\n'full' and 'calendar' produce identical weights, as documented.")

rebal = results_by_method["threshold"]
print("\nrepr:", repr(rebal))
print("trades:\n", rebal.diagnostics["trades"].round(4))
pexplain("rebalance", value=rebal)
note("rebalance")
note_explained("rebalance")

threshold  n_traded=5  turnover=42.33%  est_cost=0.0423%
full       n_traded=5  turnover=42.33%  est_cost=0.0423%
calendar   n_traded=5  turnover=42.33%  est_cost=0.0423%

'full' and 'calendar' produce identical weights, as documented.

repr: ModelResult(name='rebalance', n_assets=5, n_active=4)
trades:
 AAPL   -0.1100
GLD     0.2488
JPM     0.1745
MSFT   -0.1133
TLT    -0.2000
Name: trade, dtype: float64
rebalance (model)

What it is:
  Turns a target allocation into an actual trade list from where the book currently sits, optionally only trading assets that have drifted past a threshold.

Formula:
  threshold: final_i = target_i if |target_i - current_i| > threshold else current_i, then renormalize to sum to 1 | full/calendar: final = target

How to read it:
  diagnostics['trades'] is the signed change per asset (positive = buy, negative = sell); diagnostics['turnover'] is the one-way fraction of the book traded, same convention as metrics.costs.turnover_from_weights.

Good vs. bad:


### 5.2 `monitor`

In [34]:
limits = {
    "max_weight": 0.40,
    "min_weight": 0.0,
    "groups": {"equity": ["AAPL", "MSFT", "JPM"], "defensive": ["TLT", "GLD"]},
    "max_group": {"equity": 0.65, "defensive": 0.5},
}
report = portfolio.models.management.monitor(limits=limits)
print(report)
pexplain("monitor", value=report)
note("monitor")
note_explained("monitor")

{'ok': True, 'n_breaches': 0, 'breaches': [], 'checked': ['AAPL', 'MSFT', 'JPM', 'TLT', 'GLD']}
monitor (function)

What it is:
  Checks a portfolio's current weights against a set of hard limits (per-asset bounds, group/sector caps, gross exposure) and reports any breaches.

Formula:
  breach if weight_i > max_weight, weight_i < min_weight, sum(weights in group) > max_group[group], or sum(abs(weights)) > max_gross

How to read it:
  An empty breaches list (ok=True) means every supplied limit is currently satisfied - monitor only checks the limits you actually pass in `limits`, it has no built-in defaults.

Good vs. bad:
  ok=True is the goal; each entry in breaches identifies exactly which limit and by how much, so you can decide whether to rebalance() back toward compliance.

Caveats:
  A point-in-time check only - it says nothing about how long a breach has persisted or how it's trending. 'max_group' requires you to supply `limits['groups']` yourself (e.g. built from Portfolio.asset

In [35]:
# Force a breach to see the report shape when something's actually wrong.
p_concentrated = Portfolio(prices, weights={"AAPL": 0.8, "MSFT": 0.05, "JPM": 0.05, "TLT": 0.05, "GLD": 0.05})
breach_report = p_concentrated.models.management.monitor(limits=limits)
print(breach_report)
assert breach_report["n_breaches"] == 2 and not breach_report["ok"]

{'ok': False, 'n_breaches': 2, 'breaches': [{'type': 'max_weight', 'asset': 'AAPL', 'value': 0.7999999999999998, 'limit': 0.4}, {'type': 'max_group', 'group': 'equity', 'value': 0.8999999999999999, 'limit': 0.65}], 'checked': ['AAPL', 'MSFT', 'JPM', 'TLT', 'GLD']}


### 5.3 `compare`

In [36]:
other = Portfolio(prices, weights=target_result.weights.to_dict(), name="Optimized")
cmp = portfolio.models.management.compare(other)
print(repr(cmp))
print("\nmode:", cmp.diagnostics["mode"])
print("metric_delta (optimized - equal-weight):\n", cmp.diagnostics["metric_delta"].round(4))
pexplain("compare", value=cmp)
note("compare")
note_explained("compare")

weights_only_cmp = portfolio.models.management.compare(target_result)
print("\n", repr(weights_only_cmp))
print("mode:", weights_only_cmp.diagnostics["mode"], "(no return history on the ModelResult side)")
assert weights_only_cmp.diagnostics["mode"] == "weights_only" 

ModelResult(name='compare', n_assets=5, n_active=4)

mode: returns
metric_delta (optimized - equal-weight):
 annualized_return     0.0889
cagr                  0.0889
calmar_ratio          0.8971
conditional_var_95   -0.0033
kurtosis             -3.0074
max_drawdown          0.0117
sharpe_ratio          0.5058
skewness             -0.6991
sortino_ratio         0.4525
total_return          0.7043
value_at_risk_95     -0.0012
volatility            0.0130
win_rate              0.0440
Name: delta, dtype: float64
compare (model)

What it is:
  Compares two portfolios, optimizer results, or raw weight vectors - weight-level always, and full metric-level whenever both sides are Portfolios with their own return history.

Formula:
  weight_delta = b - a; turnover = sum(abs(weight_delta))/2; HHI = sum(w^2); metric_delta = tearsheet_summary(b) - tearsheet_summary(a) [Portfolio vs. Portfolio only]

How to read it:
  hhi_before/hhi_after are Herfindahl concentration indices (1/N for equal-weight, 1

## 6. `ModelResult` mechanics

In [37]:
result = portfolio.models.optimize(method="mean_variance")
print("isinstance ModelResult:", isinstance(result, ModelResult))
print("has _portpy_explain_name:", hasattr(result, "_portpy_explain_name"), "->", result._portpy_explain_name)
print("_portpy_explain_value is result:", result._portpy_explain_value is result)
result.explain()

try:
    result.plot()
except NotImplementedError as exc:
    print(f"\nExpected NotImplementedError (visualization module not shipped yet): {exc}")

comparison = result.compare(other)
print("\n.compare() delegates to management.compare:", repr(comparison))

isinstance ModelResult: True
has _portpy_explain_name: True -> mean_variance
_portpy_explain_value is result: True
mean_variance (model)

What it is:
  Markowitz's original portfolio problem: pick weights that maximize expected return net of a risk penalty on variance.

Formula:
  maximize w'mu - 0.5*risk_aversion*w'Sigma*w, s.t. sum(w)=1 (+ any extra constraints)

How to read it:
  Higher risk_aversion pulls the solution toward the global minimum-variance portfolio; lower risk_aversion pulls it toward concentrating in the highest-mu assets.

Good vs. bad:
  A well-diversified solution (moderate HHI, several active positions) suggests mu/Sigma disagree enough to justify holding multiple assets; a solution collapsed into 1-2 assets often means mu is dominating an under-diversified Sigma estimate (Michaud's 'error maximization').

Caveats:
  Extremely sensitive to the expected_returns input - see expected_returns' own caveats. risk_aversion has no universal 'correct' value; it must be ca

## 7. `Portfolio.models` auto-fill mechanics

Same `_AutoFillNamespace` machinery as `Portfolio.metrics` - a bare keyword call fills in
`y`/`expected_returns`/`cov_matrix`/`current_weights`/etc. from the portfolio's own data;
any positional argument disables auto-fill entirely.

In [38]:
auto_mu = portfolio.models.estimators.expected_returns()
manual_mu = expected_returns(portfolio.asset_returns())
pd.testing.assert_series_equal(auto_mu, manual_mu)
print("auto-filled expected_returns matches the manual call exactly.")

auto_result = portfolio.models.optimization.min_variance()
print("\nbare .models.optimization.min_variance():\n", auto_result.round(3))

try:
    portfolio.models.optimization.mean_variance(auto_mu)  # one positional arg
except TypeError as exc:
    print(f"\nExpected TypeError - positional args disable auto-fill entirely: {exc}")

cap = portfolio.models.construction.TurnoverCap(max_turnover=0.1)
print("\nTurnoverCap.current_weights auto-filled from the portfolio:\n", cap.current_weights)
pd.testing.assert_series_equal(cap.current_weights, portfolio.weights)

exposed = sorted(n for n in dir(portfolio.models) if not n.startswith("_"))
print("\nPortfolio.models exposes:", exposed)

auto-filled expected_returns matches the manual call exactly.

bare .models.optimization.min_variance():
 AAPL   0.0390
MSFT   0.0840
JPM    0.1820
TLT    0.4870
GLD    0.2080
Name: weight, dtype: float64

Expected TypeError - positional args disable auto-fill entirely: mean_variance() missing 1 required positional argument: 'cov_matrix'

TurnoverCap.current_weights auto-filled from the portfolio:
 AAPL   0.2000
MSFT   0.2000
JPM    0.2000
TLT    0.2000
GLD    0.2000
Name: weight, dtype: float64

Portfolio.models exposes: ['build', 'construction', 'efficient_frontier', 'estimators', 'management', 'optimization', 'optimize']


## 8. Coverage Audit

In [39]:
EXPECTED = {
    "estimators": {"expected_returns", "covariance", "capm", "fama_french", "factor_attribution", "linear_regression", "rolling_regression"},
    "optimization": {
        "mean_variance", "min_variance", "max_sharpe", "target_return", "target_volatility",
        "optimization.efficient_frontier", "black_litterman", "hierarchical_risk_parity",
        "risk_parity", "risk_budgeting", "maximum_diversification",
    },
    "construction": {"WeightBounds", "GroupCap", "TurnoverCap", "NetExposure", "GrossExposure", "optimize", "build", "construction.efficient_frontier"},
    "management": {"rebalance", "monitor", "compare"},
}
all_expected = set().union(*EXPECTED.values())

for namespace, names in EXPECTED.items():
    missing = names - COVERED
    print(f"{namespace:14s}: {len(names) - len(missing)}/{len(names)} called" + (f"  MISSING: {sorted(missing)}" if missing else ""))

missing_covered = sorted(all_expected - COVERED)
assert not missing_covered, f"Call coverage gap: {missing_covered}"
print(f"\nEvery one of {len(all_expected)} tracked estimator/optimizer/constraint/dispatch/management names was called above.")

# Explain coverage: everything with a registered Explanation should also have been rendered
# at least once via pexplain()/.explain() - constraint dataclasses and the two dispatch
# functions (optimize/build) have no card of their own (they delegate to the model they
# dispatch to), so they're excluded from this narrower check.
explainable = all_expected - {"WeightBounds", "GroupCap", "TurnoverCap", "NetExposure", "GrossExposure", "optimize", "build", "construction.efficient_frontier"}
explainable = {n.replace("optimization.efficient_frontier", "efficient_frontier") for n in explainable}
missing_explained = sorted(explainable - EXPLAINED)
assert not missing_explained, f"Explain coverage gap: {missing_explained}"
print(f"Every one of {len(explainable)} explainable names was ALSO rendered via .explain()/pexplain() above.")

# Independent, brute-force cross-check: every registered "model" card (and the handful of
# estimator/management "function" cards used here) actually renders without raising.
model_cards = available("model")
function_cards = [n for n in available("function") if n in {"expected_returns", "covariance", "linear_regression", "rolling_regression", "factor_attribution", "monitor"}]
render_failures = []
for name in sorted(set(model_cards) | set(function_cards)):
    try:
        get(name).render()
    except Exception as exc:  # noqa: BLE001 - report literally anything
        render_failures.append((name, str(exc)))
print(f"\nBrute-force get(name).render() sanity check over {len(model_cards) + len(function_cards)} names: {len(render_failures)} failures")
assert not render_failures, f"Explanation cards that fail to render: {render_failures}"

print("=" * 70)
print("MODELS DEBUG NOTEBOOK COMPLETE")
print("=" * 70)
print(f"Portfolio tested: {portfolio.name} ({portfolio.num_assets} assets, {len(portfolio.prices)} observations)")
print(f"Names called (COVERED):    {len(COVERED)}")
print(f"Names explained (EXPLAINED): {len(EXPLAINED)}")

estimators    : 7/7 called
optimization  : 11/11 called
construction  : 8/8 called
management    : 3/3 called

Every one of 29 tracked estimator/optimizer/constraint/dispatch/management names was called above.
Every one of 21 explainable names was ALSO rendered via .explain()/pexplain() above.

Brute-force get(name).render() sanity check over 21 names: 0 failures
MODELS DEBUG NOTEBOOK COMPLETE
Portfolio tested: Debug Portfolio (5 assets, 1002 observations)
Names called (COVERED):    29
Names explained (EXPLAINED): 21
